<a href="https://colab.research.google.com/github/AjitMallav/Generative-Controls/blob/main/GenerativeControls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install torch transformers scikit-learn numpy ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.9 MB/s eta 0:00:00


In [ ]:
#!pip install bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00


In [ ]:
"""
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
from typing import List, Optional, Tuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# device verification
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
"""

In [ ]:
"""
@dataclass
class ControlAxis:
    """Represents a discovered control dimension"""
    label: str
    vector: np.ndarray
    positive_example: str
    negative_example: str
    layer_idx: int
    variance: float = 0.0

"""

In [ ]:
"""
from transformers import BitsAndBytesConfig

class SemanticControlSystem:
    def __init__(
        self,
        model_id="Qwen/Qwen2.5-3B-Instruct",
        labeler_id="Qwen/Qwen2.5-7B-Instruct",
        target_layer=18,
        device=device
    ):
        print(f"\n Initializing Hybrid System on {device}...\n")
        self.device = device
        self.target_layer = target_layer

        # Main Model
        print(f" Loading Engine: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()

        # Labeler Model
        print(f"\n Loading Labeler: {labeler_id} (4-bit compressed)...")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.labeler_tokenizer = AutoTokenizer.from_pretrained(labeler_id)
        self.labeler_model = AutoModelForCausalLM.from_pretrained(
            labeler_id,
            quantization_config=quantization_config,
            device_map="auto"
        )
        self.labeler_model.eval()
        print("Hybrid setup complete!\n")

        self.variation_cache = {}

    def generate_variations(
        self,
        prompt: str,
        num_variations: int = 2,
        temperature: float = 1.2,
        max_length: int = 200
    ) -> List[str]:
        """Generate diverse variations to explore latent space"""
        print(f"Generating {num_variations} variations...")

        variations = []
        messages = [
            {"role": "system", "content": "You are a helpful writing assistant."},
            {"role": "user", "content": prompt}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        for i in range(num_variations):
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_length,
                    temperature=temperature,
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            generated_text = self.tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            )
            generated_text = self._clean_chatty_output(generated_text)
            variations.append(generated_text.strip())
            print(f"Generated variation {i + 1}/{num_variations}")

        return variations

    def extract_activations(
        self,
        texts: List[str],
        layer_idx: Optional[int] = None
    ) -> np.ndarray:
        """Extract hidden state activations from specified layer"""
        if layer_idx is None:
            layer_idx = self.target_layer

        print(f"Extracting activations from layer {layer_idx}...")
        activations = []

        for text in texts:
            messages = [
                {"role": "system", "content": "You are a helpful writing assistant."},
                {"role": "user", "content": text}
            ]
            input_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

            with torch.no_grad():
                outputs = self.model(
                    **inputs,
                    output_hidden_states=True,
                    return_dict=True
                )

            prompt_length = inputs['input_ids'].shape[1]
            all_generated_tokens = outputs.hidden_states[layer_idx][0, prompt_length:, :]

            # Average them together (Mean Pooling)
            hidden_state = all_generated_tokens.mean(dim=0)
            activations.append(hidden_state.cpu().numpy())

        print("Activations extracted")
        return np.array(activations)

    def perform_pca(
        self,
        activations: np.ndarray,
        n_components: int = 3
    ) -> Tuple[PCA, np.ndarray]:
        """Perform PCA to find principal axes of variation"""
        print(f"Running PCA to extract {n_components} components...")

        pca = PCA(n_components=n_components)
        transformed = pca.fit_transform(activations)

        print(f"Explained variance: {[f'{v:.1%}' for v in pca.explained_variance_ratio_]}")
        return pca, transformed

    def label_axis(
        self,
        positive_text: str,
        negative_text: str
    ) -> str:
        """Label axis using TinyLlama or heuristics"""
        if self.labeler_model is not None:
            return self._label_with_local_model(positive_text, negative_text)
        return self._fallback_labeling(positive_text, negative_text)

    def _label_with_local_model(self, positive_text: str, negative_text: str) -> str:
        prompt = f"""Compare these two sentences:
Sentence 1: "{positive_text}"
Sentence 2: "{negative_text}"

Choose the ONE word from this list that best describes the main difference in style, tone, or content:
[Formality, Enthusiasm, Politeness, Detail, Urgency, Professionalism, Negativity, Confidence, Verbosity]

Output ONLY the one word."""

        messages = [
            {"role": "system", "content": "You are a linguistic expert classifying text styles."},
            {"role": "user", "content": prompt}
        ]

        input_text = self.labeler_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.labeler_tokenizer(input_text, return_tensors="pt").to(self.device)

        try:
            with torch.no_grad():
                outputs = self.labeler_model.generate(
                    **inputs,
                    max_new_tokens=5,
                    temperature=0.1,
                    do_sample=False
                )

            generated = self.labeler_tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:],
                skip_special_tokens=True
            ).strip()

            import re
            words = re.findall(r'[A-Za-z]+', generated)
            if words:
                return words[0].capitalize()

            return "Style"

        except Exception as e:
            print(f"Labeling error: {e}")
            return "Style"

    def _fallback_labeling(self, positive_text: str, negative_text: str) -> str:
        """Heuristic-based labeling"""
        pos_len = len(positive_text.split())
        neg_len = len(negative_text.split())

        if abs(pos_len - neg_len) > 15:
            return "Verbosity"
        elif abs(positive_text.count('!') - negative_text.count('!')) > 2:
            return "Enthusiasm"
        elif abs(positive_text.count('?') - negative_text.count('?')) > 2:
            return "Inquisitiveness"

        formal_words = ['however', 'therefore', 'furthermore', 'moreover']
        informal_words = ['yeah', 'cool', 'hey', 'ok']

        pos_formal = sum(1 for w in formal_words if w in positive_text.lower())
        neg_formal = sum(1 for w in formal_words if w in negative_text.lower())

        if abs(pos_formal - neg_formal) > 1:
            return "Formality"

        return "Style"

    def discover_axes(
        self,
        prompt: str,
        num_variations: int = 5,
        n_axes: int = 4
    ) -> List[ControlAxis]:
        """Full pipeline: Generate, extract, PCA, label"""
        print("\n" + "="*60)
        print("🔍 DISCOVERING CONTROL AXES")
        print("="*60 + "\n")

        # Generate variations
        variations = self.generate_variations(prompt, num_variations)

        # Extract activations
        activations = self.extract_activations(variations)

        # PCA
        pca, transformed = self.perform_pca(activations, n_components=n_axes)

        # Labels
        axes = []
        print("\n Labeling axes...")
        for i in range(n_axes):
            component_scores = transformed[:, i]
            pos_idx = np.argmax(component_scores)
            neg_idx = np.argmin(component_scores)

            pos_text = variations[pos_idx]
            neg_text = variations[neg_idx]

            label = self.label_axis(pos_text, neg_text)

            vector = pca.components_[i]

            # If the label is about length, ensure Positive = Longer
            if label.lower() in ["verbosity", "length", "detail"]:
                if len(pos_text) < len(neg_text):
                    vector = vector * -1
                    pos_text, neg_text = neg_text, pos_text

            print(f"   Axis {i+1}: {label}")

            axis = ControlAxis(
                label=label,
                vector=vector,
                positive_example=pos_text,
                negative_example=neg_text,
                layer_idx=self.target_layer
            )
            axes.append(axis)

        print("\n" + "="*60)
        print(f"Discovered {len(axes)} control axes!")
        print("="*60 + "\n")

        return axes

    def steer_generation(
        self,
        prompt: str,
        axis: ControlAxis,
        coefficient: float = 1.0,
        max_new_tokens: int = 150
    ) -> str:
        """Generate text with multi-layer activation steering"""

        # 1. 👻 GHOST HOOK CLEANUP (Crucial for live UI environments!)
        # This wipes any lingering PyTorch hooks from previous interrupted slider drags
        for layer in self.model.model.layers:
            layer._forward_hooks.clear()

        messages = [
            {"role": "system", "content": "You are a helpful writing assistant."},
            {"role": "user", "content": prompt}
        ]
        input_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        # 2. Target the middle semantic layers (14 to 20)
        layer_start = 14
        layer_end = 20
        num_layers = layer_end - layer_start + 1

        # 3. Calculate Vector
        # We slightly dampen the coefficient (0.75x) to prevent Constructive Interference on same-prompt steering
        safe_coefficient = coefficient * 0.75
        steering_vector = torch.tensor(
            (axis.vector * safe_coefficient) / num_layers,
            dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device=self.device
        )

        def steering_hook(module, input, output):
            if isinstance(output, tuple):
                hidden_states = output[0]
            else:
                hidden_states = output

            # Inject the math
            hidden_states[:, -1, :] += steering_vector

            if isinstance(output, tuple):
                return (hidden_states,) + output[1:]
            return hidden_states

        # 4. Attach fresh hooks
        handles = []
        for l in range(layer_start, layer_end + 1):
            layer = self.model.model.layers[l]
            handles.append(layer.register_forward_hook(steering_hook))

        try:
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    min_new_tokens=10,        # <--- FORCES the model to push past the EOS token!
                    repetition_penalty=1.1,   # <--- Prevents looping when forced to talk
                    do_sample=True,
                    temperature=0.3,          # <--- Slightly warmer to help it flow around the vector
                    top_p=0.9,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            generated_text = self.tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            )
        finally:
            # 5. Clean up
            for handle in handles:
                handle.remove()

        return generated_text.strip()

print("SemanticControlSystem class defined!")
"""

In [ ]:
"""
!pip install -U bitsandbytes>=0.46.1 -q
system = SemanticControlSystem(
    model_id="Qwen/Qwen2.5-3B-Instruct",
    labeler_id="Qwen/Qwen2.5-7B-Instruct"
)
"""

In [ ]:
"""
prompt = "Write a one-sentence text message to my friend canceling on their party tonight."

axes = system.discover_axes(
    prompt=prompt,
    num_variations=5,
    n_axes=4
)

print("\n DISCOVERED AXES:\n")
for i, axis in enumerate(axes, 1):
    display_label = axis.label if axis.label != "Text" else f"Style {i}"

    print(f"{i}. {display_label}")
    print(f"Positive: {axis.positive_example}")
    print(f"Negative: {axis.negative_example}")
    print()
"""

In [ ]:
"""
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Gemini suggestion: Remove any ghost hooks that might be stuck in the model
for layer in system.model.model.layers:
    layer._forward_hooks.clear()

prompt_input = widgets.Textarea(
    value="Write a one-sentence text message to my friend canceling on their party tonight.",
    description="Prompt:",
    layout=widgets.Layout(width='80%', height='60px')
)

# makes sure 1 axis at a time
axis_dropdown = widgets.Dropdown(
    options=[(f"Axis {i+1}: {axis.label}", i) for i, axis in enumerate(axes)],
    value=0,
    description="Concept:",
    layout=widgets.Layout(width='50%')
)

slider = widgets.FloatSlider(
    value=0.0,
    min=-15.0,
    max=15.0,
    step=1.0,
    description="Strength:",
    continuous_update=False,
    layout=widgets.Layout(width='80%')
)

output_area = widgets.Output()

def on_generate_click(b):
    with output_area:
        clear_output()

        # Dropdown for axes
        active_axis = axes[axis_dropdown.value]
        active_coeff = slider.value

        print(f"Steering '{active_axis.label}' to {active_coeff:+}...\n")
        print("-" * 70)

        # live generation
        result = system.steer_generation(
            prompt=prompt_input.value,
            axis=active_axis,
            coefficient=active_coeff,
            max_new_tokens=250
        )

        print(f"(Positive Direction): {active_axis.positive_example[:100]}...")
        print(f"(Negative Direction): {active_axis.negative_example[:100]}...\n")
        print(f"Steered Output:\n{result}")

generate_btn = widgets.Button(
    description="Generate",
    button_style="success",
    layout=widgets.Layout(width='200px')
)
generate_btn.on_click(on_generate_click)

# UI
display(HTML("<h3 style='color: #2c3e50;'>Single-Axis Control Panel</h3>"))
display(prompt_input)
display(axis_dropdown)
display(slider)
display(generate_btn)
display(output_area)
"""

In [ ]:
"""
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

def visualize_latent_space(system, prompt, num_variations=10):
    print("Waiting...")

    variations = system.generate_variations(prompt, num_variations, temperature=1.5)
    activations = system.extract_activations(variations)

    pca, transformed = system.perform_pca(activations, n_components=3)

    df = pd.DataFrame({
        'PC1 (Axis 1)': transformed[:, 0],
        'PC2 (Axis 2)': transformed[:, 1],
        'PC3 (Axis 3)': transformed[:, 2],
        'Text': [v[:60] + "..." for v in variations],
        'Full_Text': variations
    })

    fig = px.scatter_3d(
        df, x='PC1 (Axis 1)', y='PC2 (Axis 2)', z='PC3 (Axis 3)',
        hover_name='Text',
        title="3D Latent Space Visualization of Model Generations",
        color='PC1 (Axis 1)',
        color_continuous_scale='Viridis'
    )

    fig.update_traces(marker=dict(size=8, line=dict(width=2, color='DarkSlateGrey')))

    fig.add_trace(go.Scatter3d(
        x=[df['PC1 (Axis 1)'].min(), df['PC1 (Axis 1)'].max()],
        y=[0, 0],
        z=[0, 0],
        mode='lines',
        line=dict(color='red', width=5),
        name='Steering Vector (Axis 1)'
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title='Axis 1 (Highest Variance)',
            yaxis_title='Axis 2',
            zaxis_title='Axis 3'
        ),
        width=900,
        height=700,
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

visualize_latent_space(system, "Write a one-sentence text message to my friend canceling on their party tonight.")
"""

In [ ]:
"""
# Install server and tunneling libraries
!pip install fastapi uvicorn nest-asyncio pydantic -q
!npm install -g localtunnel -q

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import threading
import subprocess
import time

# Define the data structures
class DiscoverRequest(BaseModel):
    prompt: str
    num_variations: int = 5
    n_axes: int = 4

class SteerRequest(BaseModel):
    prompt: str
    axis_index: int
    coefficient: float
    max_tokens: int = 150

# Initialize the API
app = FastAPI(title="Latent-Steer API")

# Allow the React app to talk to this Colab server
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

global_axes = []

# NEW: Dictionary to store previously generated text
# Format: {(prompt, axis_index, coefficient): "Generated Text"}
generation_cache = {}

# Define the Endpoints
@app.post("/discover")
async def discover_endpoint(req: DiscoverRequest):
    global global_axes
    global generation_cache

    print(f"\n API Request: Discovering axes for -> '{req.prompt}'")

    # Clear the old generation cache when a completely new prompt is sent
    # This prevents the server RAM from overflowing during long testing sessions
    generation_cache.clear()

    global_axes = system.discover_axes(
        prompt=req.prompt,
        num_variations=req.num_variations,
        n_axes=req.n_axes
    )

    response_data = []
    for i, axis in enumerate(global_axes):
        response_data.append({
            "index": i,
            "label": axis.label,
            "positive_example": axis.positive_example,
            "negative_example": axis.negative_example
            "variance": float(pca.explained_variance_ratio_[i])
        })

    return {"status": "success", "axes": response_data}

@app.post("/steer")
async def steer_endpoint(req: SteerRequest):
    global generation_cache

    if not global_axes or req.axis_index >= len(global_axes):
        return {"error": "Invalid axis. Please run /discover first."}

    target_axis = global_axes[req.axis_index]

    # 1. Round the coefficient to 1 decimal to prevent floating-point jitter from the React slider
    rounded_coef = round(req.coefficient, 1)

    # 2. Create the unique cache key
    cache_key = (req.prompt, req.axis_index, rounded_coef)

    # 3. CHECK THE CACHE FIRST
    if cache_key in generation_cache:
        print(f"\n ⚡ CACHE HIT: Returning instant saved text for {target_axis.label} at {rounded_coef}")
        return {"status": "success", "generated_text": generation_cache[cache_key]}

    # 4. If not in cache, we must use the GPU
    print(f"\n API Request: Steering {target_axis.label} to {rounded_coef}")

    generated_text = system.steer_generation(
        prompt=req.prompt,
        axis=target_axis,
        coefficient=rounded_coef,
        max_new_tokens=req.max_tokens
    )

    # 5. SAVE TO CACHE for next time
    generation_cache[cache_key] = generated_text

    return {"status": "success", "generated_text": generated_text}

# Launch the Server
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server)
server_thread.start()

# Create a public tunnel to bypass VS Code port forwarding restrictions
print("Creating public tunnel...")
time.sleep(3)
process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
for line in process.stdout:
    print(f"\n YOUR PUBLIC API URL: {line.decode().strip()}\n")
    break
"""

In [ ]:
# !curl ipv4.icanhazip.com

In [ ]:
"""
import requests
url = "http://127.0.0.1:8000"
print(requests.get(url + "/docs").status_code)
""

#**New**


In [1]:
!pip uninstall -y sympy
!pip install -q --no-cache-dir --force-reinstall "sympy==1.13.1"
!pip install -q --no-cache-dir --force-reinstall "mpmath==1.3.0"

Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 447.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires sympy>=1.13.3, but you have sympy 1.13.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires sympy>=1.13.3, but you have sympy 1.13.1 which is incompatible.


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import sympy
print("sympy version:", sympy.__version__)
print("sympy path:", sympy.__file__)
print("has sympy.printing:", hasattr(sympy, "printing"))

import sympy.printing
print("sympy.printing import worked")

sympy version: 1.13.1
sympy path: /usr/local/lib/python3.12/dist-packages/sympy/__init__.py
has sympy.printing: True
sympy.printing import worked


In [2]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate transformers nest-asyncio fastapi uvicorn pydantic scikit-learn
!npm install -g localtunnel -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 124.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 3s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [3]:
from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Tuple
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.decomposition import PCA

# -----------------------------
# Device verification
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


@dataclass
class ControlAxis:
    label: str
    vector: np.ndarray
    positive_example: str
    negative_example: str
    layer_idx: int
    variance: float = 0.0


class SemanticControlSystem:
    def __init__(
        self,
        model_id: str = "Qwen/Qwen2.5-3B-Instruct",
        labeler_id: Optional[str] = None,
        target_layer: int = 18,
        device: str = device
    ):
        print(f"\nInitializing Hybrid System on {device}...\n")

        self.device = device
        self.target_layer = target_layer
        self.validation_coefficients = [-2, -1, -0.5, 0, 0.5, 1, 2]

        # -----------------------------
        # Main engine model
        # -----------------------------
        print(f"Loading Engine: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto"
        )
        self.model.eval()

        # -----------------------------
        # Labeler model
        # -----------------------------
        if labeler_id is None:
            print("\nUsing Engine model as Labeler to save VRAM...")
            self.labeler_tokenizer = self.tokenizer
            self.labeler_model = self.model
        else:
            print(f"\nLoading Labeler: {labeler_id}...")

            labeler_kwargs = {"device_map": "auto"}

            if self.device == "cuda":
                quantization_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16
                )
                labeler_kwargs["quantization_config"] = quantization_config
            else:
                labeler_kwargs["torch_dtype"] = torch.float32

            self.labeler_tokenizer = AutoTokenizer.from_pretrained(labeler_id)

            if self.labeler_tokenizer.pad_token is None:
                self.labeler_tokenizer.pad_token = self.labeler_tokenizer.eos_token

            self.labeler_model = AutoModelForCausalLM.from_pretrained(
                labeler_id,
                **labeler_kwargs
            )
            self.labeler_model.eval()

        print("Hybrid setup complete!\n")

    # ============================================================
    # 1. Generation-time activation extraction
    # ============================================================

    def _generate_with_generation_activations(
        self,
        prompt: str,
        system_prompt: str,
        layer_idx: Optional[int] = None,
        max_new_tokens: int = 60,
        temperature: float = 0.9,
        top_p: float = 0.95,
        repetition_penalty: float = 1.15
    ) -> Tuple[str, np.ndarray]:
        """
        Generates text and captures hidden states during the actual generation process.

        PCA now uses generation-time hidden states, not hidden states from re-reading
        the completed response afterward.
        """
        if layer_idx is None:
            layer_idx = self.target_layer

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)
        prompt_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                pad_token_id=self.tokenizer.eos_token_id,
                return_dict_in_generate=True,
                output_hidden_states=True
            )

        generated_ids = outputs.sequences[0][prompt_len:]
        generated_text = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()
        generated_text = self._clean_chatty_output(generated_text)

        token_vectors = []

        if outputs.hidden_states is not None:
            for step_hidden_states in outputs.hidden_states:
                layer_hidden = step_hidden_states[layer_idx][0, -1, :]
                token_vectors.append(layer_hidden.detach().float().cpu())

        if len(token_vectors) == 0:
            full_ids = outputs.sequences[:, :]
            with torch.no_grad():
                forward_out = self.model(
                    input_ids=full_ids,
                    output_hidden_states=True,
                    return_dict=True
                )

            hidden = forward_out.hidden_states[layer_idx][0, prompt_len:, :]

            if hidden.shape[0] == 0:
                response_vector = forward_out.hidden_states[layer_idx][0, -1, :]
            else:
                response_vector = hidden.mean(dim=0)

            response_vector = response_vector.detach().float().cpu().numpy()
        else:
            response_vector = torch.stack(token_vectors, dim=0).mean(dim=0).numpy()

        return generated_text, response_vector

    # ============================================================
    # 2. Generate variation cloud
    # ============================================================

    def generate_variations_with_activations(
        self,
        prompt: str,
        num_variations: int = 20,
        layer_idx: Optional[int] = None
    ) -> Tuple[List[str], np.ndarray]:
        """
        Generates diverse responses and returns:
        - variation texts
        - generation-time activation vectors for each response
        """
        if layer_idx is None:
            layer_idx = self.target_layer

        variations = []
        activations = []

        system_prompts = [
            "You are a standard helpful assistant. Give a natural response.",
            "You are a helpful assistant. Make the response slightly more formal.",
            "You are a helpful assistant. Make the response slightly more casual.",
            "You are a helpful assistant. Make the response concise but complete.",
            "You are a helpful assistant. Make the response somewhat more detailed.",
            "You are a helpful assistant. Make the response more vivid and expressive.",
            "You are a helpful assistant. Make the response direct and plainspoken.",
            "You are a helpful assistant. Make the response polished and professional.",
        ]

        for i in range(num_variations):
            sys_msg = system_prompts[i % len(system_prompts)]

            text, vec = self._generate_with_generation_activations(
                prompt=prompt,
                system_prompt=sys_msg,
                layer_idx=layer_idx,
                max_new_tokens=70,
                temperature=0.9,
                top_p=0.95,
                repetition_penalty=1.15
            )

            if text.strip():
                variations.append(text)
                activations.append(vec)
                print(f"[Variation {len(variations)}]: {text[:80]}...")

        if len(variations) < 3:
            raise ValueError("Not enough valid variations were generated for PCA.")

        return variations, np.array(activations)

    def generate_variations(self, prompt: str, num_variations: int = 20) -> List[str]:
        variations, _ = self.generate_variations_with_activations(
            prompt=prompt,
            num_variations=num_variations,
            layer_idx=self.target_layer
        )
        return variations

    def extract_activations(
        self,
        texts: List[str],
        layer_idx: Optional[int] = None
    ) -> np.ndarray:
        """
        Legacy fallback.

        discover_axes does NOT use this anymore.
        It now uses generation-time activations instead.
        """
        if layer_idx is None:
            layer_idx = self.target_layer

        activations = []

        for text in texts:
            messages = [
                {"role": "system", "content": "You are a helpful writing assistant."},
                {"role": "user", "content": text}
            ]

            input_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

            with torch.no_grad():
                outputs = self.model(
                    **inputs,
                    output_hidden_states=True,
                    return_dict=True
                )

            hidden_state = outputs.hidden_states[layer_idx][0, -1, :]
            activations.append(hidden_state.detach().float().cpu().numpy())

        return np.array(activations)

    # ============================================================
    # 3. PCA axis discovery
    # ============================================================

    def discover_axes(
        self,
        prompt: str,
        num_variations: int = 20,
        target_axes: int = 3,
        variance_threshold: float = 0.05
    ):
        """
        Main PCA discovery method.

        Pipeline:
        prompt -> diverse generations -> generation-time activations -> PCA
        -> endpoint examples -> label -> polarity align -> axes.
        """
        variations, activations = self.generate_variations_with_activations(
            prompt=prompt,
            num_variations=num_variations,
            layer_idx=self.target_layer
        )

        max_components = min(len(variations), target_axes + 4, activations.shape[1])

        pca = PCA(n_components=max_components)
        transformed = pca.fit_transform(activations)

        axes = []
        seen_label_roots = []

        for i in range(max_components):
            if len(axes) >= target_axes:
                break

            variance = float(pca.explained_variance_ratio_[i])

            if variance < variance_threshold:
                print(
                    f"[Filter] Dropped PC{i}: variance {variance * 100:.1f}% below threshold."
                )
                continue

            component_scores = transformed[:, i]
            pos_idx = int(np.argmax(component_scores))
            neg_idx = int(np.argmin(component_scores))

            pos_text = variations[pos_idx]
            neg_text = variations[neg_idx]

            print("\nAXIS DEBUG:")
            print(f"PC: {i}")
            print(f"Variance: {variance:.4f}")
            print("POS:", pos_text)
            print("NEG:", neg_text)

            label = self._label_with_local_model(pos_text, neg_text)
            print("Label:", label)

            root = label[:4].lower()
            if root in seen_label_roots:
                print(f"[Filter] Dropped redundant axis: '{label}'")
                continue

            seen_label_roots.append(root)

            raw_vector = pca.components_[i]
            vector_tensor = torch.tensor(raw_vector, dtype=torch.float32)
            vector_normalized = F.normalize(vector_tensor, p=2, dim=0).numpy()

            is_intuitively_aligned = self._align_polarity(label, pos_text, neg_text)

            is_length_inversion = (
                label.lower() in ["verbosity", "length", "detail", "conciseness"]
                and len(pos_text) < len(neg_text)
            )

            if not is_intuitively_aligned or is_length_inversion:
                vector_normalized = vector_normalized * -1
                pos_text, neg_text = neg_text, pos_text

            axis = ControlAxis(
                label=label,
                vector=vector_normalized,
                positive_example=self._clean_chatty_output(pos_text),
                negative_example=self._clean_chatty_output(neg_text),
                layer_idx=self.target_layer,
                variance=variance
            )

            axis.index = len(axes)
            axes.append(axis)

        print(f"[Discovery Complete] Yielded {len(axes)} axes.")
        return axes, variations

    # ============================================================
    # 4. Optional custom axis, not the core PCA method
    # ============================================================

    def extract_custom_axis(
        self,
        concept: str,
        layer_idx: Optional[int] = None
    ) -> ControlAxis:
        """
        Optional compatibility feature.

        Not the core research method.
        PCA-discovered axes remain the main contribution.
        """
        if layer_idx is None:
            layer_idx = self.target_layer

        print(f"\n[Custom Axis] Synthesizing contrast for: {concept}")

        baseline_prompt = "Write exactly two sentences describing a walk in the park."

        pos_sys = (
            f"You are a helpful assistant. Write in a style that is extremely {concept}."
        )
        neg_sys = (
            f"You are a helpful assistant. Write in a style that is the opposite of {concept}."
        )

        pos_text, pos_vec = self._generate_with_generation_activations(
            prompt=baseline_prompt,
            system_prompt=pos_sys,
            layer_idx=layer_idx,
            max_new_tokens=70,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )

        neg_text, neg_vec = self._generate_with_generation_activations(
            prompt=baseline_prompt,
            system_prompt=neg_sys,
            layer_idx=layer_idx,
            max_new_tokens=70,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )

        raw_vector = pos_vec - neg_vec
        vector_tensor = torch.tensor(raw_vector, dtype=torch.float32)
        vector_normalized = F.normalize(vector_tensor, p=2, dim=0).numpy()

        return ControlAxis(
            label=concept.capitalize(),
            vector=vector_normalized,
            positive_example=pos_text,
            negative_example=neg_text,
            layer_idx=layer_idx,
            variance=1.0
        )

    # ============================================================
    # 5. Steering
    # ============================================================

    def steer_generation(
        self,
        prompt: str,
        axis: ControlAxis,
        coefficient: float = 1.0,
        max_new_tokens: int = 150
    ) -> str:
        """
        Steers generation by injecting the PCA vector into the same layer used
        to discover the axis.

        Important fixes:
        - injects at axis.layer_idx
        - does not divide by number of layers
        - no brittle token filtering
        """
        messages = [
            {"role": "system", "content": "You are a helpful writing assistant."},
            {"role": "user", "content": prompt}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        steering_vector = torch.tensor(
            axis.vector * coefficient,
            dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device=self.device
        )

        target_layer = axis.layer_idx

        def steering_hook(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            hidden_states = hidden_states.clone()

            hidden_states[:, -1, :] += steering_vector

            return (hidden_states,) + output[1:] if isinstance(output, tuple) else hidden_states

        handle = self.model.model.layers[target_layer].register_forward_hook(steering_hook)

        try:
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    min_new_tokens=5,
                    repetition_penalty=1.1,
                    do_sample=True,
                    temperature=0.4,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            generated_text = self.tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            ).strip()

            generated_text = self._clean_chatty_output(generated_text)

            if not generated_text:
                generated_text = "[Model generated whitespace/silence. Try a different coefficient.]"

        finally:
            handle.remove()

        return generated_text

    # ============================================================
    # 6. Validation sweep
    # ============================================================

    def validate_axis_sweep(
        self,
        prompt: str,
        axis: ControlAxis,
        coefficients: Optional[List[float]] = None,
        max_new_tokens: int = 150,
        run_judge: bool = True
    ) -> Dict[str, Any]:
        """
        Generates outputs across a coefficient sweep:
        [-2, -1, -0.5, 0, 0.5, 1, 2]

        This lets you inspect whether the axis behaves monotonically.
        """
        if coefficients is None:
            coefficients = self.validation_coefficients

        results = []
        text_by_coef = {}

        for coef in coefficients:
            print(f"[Validation] Axis '{axis.label}' at coefficient {coef}")

            generated = self.steer_generation(
                prompt=prompt,
                axis=axis,
                coefficient=coef,
                max_new_tokens=max_new_tokens
            )

            text_by_coef[coef] = generated

            results.append({
                "coefficient": coef,
                "generated_text": generated
            })

        judgment = None

        if (
            run_judge
            and -1 in text_by_coef
            and 0 in text_by_coef
            and 1 in text_by_coef
        ):
            judgment = self.judge_steering_direction(
                concept=axis.label,
                negative_text=text_by_coef[-1],
                baseline_text=text_by_coef[0],
                positive_text=text_by_coef[1]
            )

        return {
            "axis_label": axis.label,
            "axis_index": getattr(axis, "index", None),
            "coefficients": coefficients,
            "sweep_results": results,
            "judge_summary": judgment
        }

    def validate_axes(
        self,
        prompt: str,
        axes: List[ControlAxis],
        coefficients: Optional[List[float]] = None,
        max_new_tokens: int = 150,
        run_judge: bool = True
    ) -> List[Dict[str, Any]]:
        results = []

        for axis in axes:
            results.append(
                self.validate_axis_sweep(
                    prompt=prompt,
                    axis=axis,
                    coefficients=coefficients,
                    max_new_tokens=max_new_tokens,
                    run_judge=run_judge
                )
            )

        return results

    def judge_steering_direction(
        self,
        concept: str,
        negative_text: str,
        baseline_text: str,
        positive_text: str
    ) -> Dict[str, Any]:
        """
        Uses the labeler to check whether:
        negative < baseline < positive
        for the named concept.
        """
        prompt = (
            "You are an expert evaluator of stylistic and semantic attributes in text.\n"
            f"Concept: {concept}\n\n"
            "Below are three texts intended to represent increasing amounts of the concept.\n\n"
            f"Text A:\n{negative_text}\n\n"
            f"Text B:\n{baseline_text}\n\n"
            f"Text C:\n{positive_text}\n\n"
            "Q1: Is Text C MORE of the concept than Text B? Answer only YES or NO.\n"
            "Q2: Is Text A LESS of the concept than Text B? Answer only YES or NO.\n"
            "Q3: In one short sentence, what changed?"
        )

        messages = [
            {"role": "system", "content": "You are a concise expert evaluator."},
            {"role": "user", "content": prompt}
        ]

        input_text = self.labeler_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.labeler_tokenizer(input_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.labeler_model.generate(
                **inputs,
                max_new_tokens=60,
                do_sample=False,
                temperature=0.1,
                pad_token_id=self.labeler_tokenizer.eos_token_id
            )

        raw = self.labeler_tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        raw_upper = raw.upper()
        yes_count = raw_upper.count("YES")
        ordered = yes_count >= 2

        return {
            "ordered_correctly": ordered,
            "raw_judgment": raw
        }

    # ============================================================
    # 7. Labeling and polarity
    # ============================================================

    def _label_with_local_model(self, pos: str, neg: str) -> str:
        """
        Labels the difference between PCA endpoints.

        Returns one compact UI label.
        """
        prompt = (
            "You are an expert linguistic annotator.\n"
            "Read two texts and output EXACTLY ONE short label describing the main stylistic or semantic difference.\n"
            "The label should be 1 to 2 words maximum.\n\n"
            "Text 1: \"Yeah I can't make it bro.\"\n"
            "Text 2: \"I sincerely apologize, but I am unable to attend.\"\n"
            "Label: Formality\n\n"
            "Text 1: \"I went to the store.\"\n"
            "Text 2: \"I hurriedly dashed to the supermarket!\"\n"
            "Label: Urgency\n\n"
            "Text 1: \"The sky is blue.\"\n"
            "Text 2: \"The sky is a brilliant, breathtaking sapphire!\"\n"
            "Label: Descriptiveness\n\n"
            f"Text 1: \"{pos}\"\n"
            f"Text 2: \"{neg}\"\n"
            "Label:"
        )

        messages = [
            {"role": "system", "content": "You are a concise linguistic judge."},
            {"role": "user", "content": prompt}
        ]

        input_text = self.labeler_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.labeler_tokenizer(input_text, return_tensors="pt").to(self.device)

        try:
            with torch.no_grad():
                outputs = self.labeler_model.generate(
                    **inputs,
                    max_new_tokens=8,
                    temperature=0.1,
                    do_sample=False,
                    pad_token_id=self.labeler_tokenizer.eos_token_id
                )

            generated = self.labeler_tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            ).strip()

            print("\n[Labeler Audit]")
            print(f"Pos: {pos[:80]}...")
            print(f"Neg: {neg[:80]}...")
            print(f"Raw Labeler Output: '{generated}'")

            import re
            words = re.findall(r"[A-Za-z]+", generated)

            if not words:
                return "Style"

            if len(words) >= 2:
                final_label = f"{words[0].capitalize()} {words[1].capitalize()}"
            else:
                final_label = words[0].capitalize()

            print(f"Clean UI Label: {final_label}")
            return final_label

        except Exception as e:
            print(f"[Labeler Error]: {e}")
            return "Style"

    def _align_polarity(
        self,
        label: str,
        pos_text: str,
        neg_text: str
    ) -> bool:
        """
        Checks whether pos_text has more of the labeled concept than neg_text.

        Returns True if PCA positive endpoint is aligned.
        """
        prompt = (
            "You are an expert judge.\n"
            "Which text has MORE of the concept?\n"
            "Output strictly the number 1 or 2.\n\n"
            "Text 1: \"Yeah whatever bro\"\n"
            "Text 2: \"I deeply apologize for the inconvenience.\"\n"
            "Concept: Formality\n"
            "Answer: 2\n\n"
            f"Text 1: \"{pos_text}\"\n"
            f"Text 2: \"{neg_text}\"\n"
            f"Concept: {label}\n"
            "Answer:"
        )

        messages = [
            {"role": "system", "content": "You are an expert linguistic judge."},
            {"role": "user", "content": prompt}
        ]

        input_text = self.labeler_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.labeler_tokenizer(input_text, return_tensors="pt").to(self.device)

        try:
            with torch.no_grad():
                outputs = self.labeler_model.generate(
                    **inputs,
                    max_new_tokens=2,
                    temperature=0.1,
                    do_sample=False,
                    pad_token_id=self.labeler_tokenizer.eos_token_id
                )

            generated = self.labeler_tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            ).strip()

            return "1" in generated

        except Exception:
            return True

    # ============================================================
    # 8. Cleaning
    # ============================================================

    def _clean_chatty_output(self, text: str) -> str:
        import re

        text = text.strip()

        patterns = [
            r"^Sure[,\!]?\s*",
            r"^Here(?:'s| is)\s+(?:a|the)?\s*.*?:\s*",
            r"^Response:\s*",
            r"^Output:\s*",
        ]

        for pattern in patterns:
            text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

        match = re.search(r':\s*"', text)
        if match:
            text = text[match.end():].strip()

        if text.startswith('"') and text.endswith('"'):
            text = text[1:-1].strip()
        elif text.endswith('"'):
            text = text[:-1].strip()

        return text


print("SemanticControlSystem class defined with generation-time PCA steering and validation sweep support!")

Using device: cuda
GPU: Tesla T4
SemanticControlSystem class defined with generation-time PCA steering and validation sweep support!


In [4]:
# Initialize the system and load weights into GPU memory
system = SemanticControlSystem()


Initializing Hybrid System on cuda...

Loading Engine: Qwen/Qwen2.5-3B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Using Engine model as Labeler to save VRAM...
Hybrid setup complete!



In [5]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import threading
import subprocess
import time

app = FastAPI(title="Latent-Steer API")

app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"],
)

global_axes = []
generation_cache = {}

class DiscoverRequest(BaseModel):
    prompt: str
    num_variations: int = 10
    n_axes: int = 3

class SteerRequest(BaseModel):
    prompt: str
    axis_index: int
    coefficient: float
    max_tokens: int = 250

class CustomAxisRequest(BaseModel):
    concept: str

@app.post("/discover")
async def discover_endpoint(req: DiscoverRequest):
    global global_axes
    global generation_cache
    print(f"\n API Request: Discovering axes for -> '{req.prompt}'")
    generation_cache.clear()

    # 1. Unpack BOTH the axes and the variations from the tuple
    global_axes, variations = system.discover_axes(
        prompt=req.prompt,
        num_variations=req.num_variations,
        target_axes=req.n_axes # Mapped to your new Python argument name!
    )

    baseline_text = system.steer_generation(
        prompt=req.prompt,
        axis=global_axes[0],
        coefficient=0.0,
        max_new_tokens=100
    )

    response_data = [{"index": i, "label": ax.label, "positive_example": ax.positive_example, "negative_example": ax.negative_example, "variance": ax.variance} for i, ax in enumerate(global_axes)]

    # 2. Add the variations list to the JSON response so React can render the Cloud Explorer
    return {
        "status": "success",
        "axes": response_data,
        "variations": variations,
        "baseline": baseline_text
    }

@app.post("/custom_axis")
async def custom_axis_endpoint(req: CustomAxisRequest):
    global global_axes
    print(f"\n API Request: Creating custom axis for -> '{req.concept}'")

    custom_axis = system.extract_custom_axis(concept=req.concept)
    new_index = len(global_axes)
    global_axes.append(custom_axis)

    return {
        "status": "success",
        "axis": {
            "index": new_index,
            "label": custom_axis.label,
            "positive_example": custom_axis.positive_example,
            "negative_example": custom_axis.negative_example,
            "variance": custom_axis.variance
        }
    }

@app.post("/steer")
async def steer_endpoint(req: SteerRequest):
    global generation_cache
    if not global_axes or req.axis_index >= len(global_axes):
        return {"error": "Invalid axis. Please run /discover first."}

    target_axis = global_axes[req.axis_index]
    rounded_coef = round(req.coefficient, 1)
    cache_key = (req.prompt, req.axis_index, rounded_coef)

    if cache_key in generation_cache:
        print(f"\n ⚡ CACHE HIT: Returning instant saved text for {target_axis.label} at {rounded_coef}")
        return {"status": "success", "generated_text": generation_cache[cache_key]}

    print(f"\n API Request: Steering {target_axis.label} to {rounded_coef}")
    generated_text = system.steer_generation(prompt=req.prompt, axis=target_axis, coefficient=rounded_coef, max_new_tokens=req.max_tokens)

    generation_cache[cache_key] = generated_text
    return {"status": "success", "generated_text": generated_text}

# Launch the Server
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server)
server_thread.start()

print("Creating public tunnel...")
time.sleep(3)
process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
for line in process.stdout:
    print(f"\n YOUR PUBLIC API URL: {line.decode().strip()}\n")
    break

Creating public tunnel...

 YOUR PUBLIC API URL: your url is: https://light-points-push.loca.lt



In [ ]:
!wget -q -O - ipv4.icanhazip.com

34.182.102.235
